<a href="https://colab.research.google.com/github/LeNhan18/SystemTrafficLaw/blob/main/ObjectTracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.6 MB/s eta 0:00:00


In [3]:
import os
dataset_root = '/content/drive/MyDrive/Colab Notebooks/DatasetTransportation/Transportation'
data_path = '/content/drive/MyDrive/Colab Notebooks/DatasetTransportation/Transportation/data.yaml'
if not os.path.exists(dataset_root):
  print('Folder không tồn tại')
else:
  print('Folder tồn tại')

  required_dirs = ['images/train','label/train','images/val','label/val']
  missing = []
  for dir in required_dirs:
    full_path = os.path.join(dataset_root,dir)
    if not os.path.exists(full_path):
      missing.append(dir)


Folder tồn tại


In [4]:
from ultralytics import YOLO
import ultralytics
ultralytics.checks()
model = YOLO('yolov8n.pt')
results = model.train(
    data = (data_path),
    epochs =100,
    imgsz = 640,
    batch = 16,
    hsv_h=  0.015, hsv_s=  0.7, hsv_v=0.4, # Tăng biến thiên value cho mưa sáng
    name ='vehicle_tracking_helmet_violation',
    patience = 20,

)

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.5/112.6 GB disk)
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/DatasetTransportation/Transportation/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

In [5]:
from ultralytics import YOLO
import cv2
import numpy as np
from google.colab import files
#Load  model vừa train
model_path = "/content/runs/detect/vehicle_tracking_helmet_violation/weights/best.pt"
model = YOLO(model_path)

# Upload hoặc dùng video từ Drive
# uploaded = files.upload()  # Nếu upload từ máy
# video_path = list(uploaded.keys())[0]

video_path = "/content/drive/MyDrive/traffic_test.mp4"  # thay bằng đường dẫn video mưa sáng của bạn

# Hàm IoU
def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

# Tracking + logic violation
cap = cv2.VideoCapture(video_path)
frame_count = 0
output_path = "/content/tracked_violation.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), 30, (int(cap.get(3)), int(cap.get(4))))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run tracking
    results = model.track(frame, persist=True, tracker="botsort.yaml", conf=0.4, iou=0.5)

    person_list, head_list, helmet_list, moto_list = [], [], [], []

    for r in results:
        for box in r.boxes:
            if box.id is None: continue
            cls = int(box.cls)
            xyxy = box.xyxy[0].cpu().numpy().astype(int)
            track_id = int(box.id)

            if cls == 0:   # person
                person_list.append({"box": xyxy, "id": track_id})
            elif cls == 1: # head
                head_list.append({"box": xyxy, "id": track_id})
            elif cls == 2: # helmet
                helmet_list.append({"box": xyxy, "id": track_id})
            elif cls == 3: # motorcycle (hoặc class xe máy của bạn)
                moto_list.append({"box": xyxy, "id": track_id})

    # Association và check violation
    for moto in moto_list:
        moto_box = moto["box"]
        cv2.rectangle(frame, (moto_box[0], moto_box[1]), (moto_box[2], moto_box[3]), (255, 0, 0), 2)
        cv2.putText(frame, f"XE ID: {moto['id']}", (moto_box[0], moto_box[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,0,0), 2)

        for person in person_list:
            if calculate_iou(moto_box, person["box"]) > 0.25:  # rider trên xe
                has_helmet = False
                for head in head_list:
                    if calculate_iou(person["box"], head["box"]) > 0.3:
                        for helmet in helmet_list:
                            if calculate_iou(head["box"], helmet["box"]) > 0.4:
                                has_helmet = True
                                break
                        if not has_helmet:
                            cv2.rectangle(frame, (person["box"][0], person["box"][1]), (person["box"][2], person["box"][3]), (0,0,255), 2)
                            cv2.putText(frame, f"VI PHAM - ID: {person['id']}", (person["box"][0], person["box"][1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)

    out.write(frame)
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Đã xử lý {frame_count} frames")

cap.release()
out.release()
print("Video output lưu tại:", output_path)

# Download video
files.download(output_path)

Video output lưu tại: /content/tracked_violation.mp4


FileNotFoundError: Cannot find file: /content/tracked_violation.mp4